# Module 03: The Butterfly -- Fast Walsh-Hadamard Transform

**Time**: ~30 minutes

**Prerequisites**: You know that the FFT decomposes signals into frequencies, and you just built a Lloyd-Max quantizer in Module 02. No prior transform implementation experience needed.

**What you will learn**:
1. How matrix multiplication transforms vectors
2. The Hadamard matrix and its recursive structure
3. Why spreading information across coordinates matters for quantization
4. The butterfly trick: from O(d^2) to O(d log d)
5. A working FWHT implementation from scratch

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_close, check_shape, norm_preservation_test, section_header

import math
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
## 1. Matrices Transform Vectors

The simplest way to transform a vector is to multiply it by a matrix. Different matrices do different things:

- The **identity matrix** does nothing: $I \mathbf{x} = \mathbf{x}$
- A **rotation matrix** rotates the vector in space
- A **scaling matrix** stretches or shrinks along each axis

Let's see this concretely in 2D.

In [ ]:
section_header("Matrix Multiplication = Transformation")

x = torch.tensor([1.0, 0.0])  # a simple 2D vector

# Identity: does nothing
I = torch.eye(2)
print(f"Identity:  I @ x = {I @ x}")

# Rotation by 45 degrees
angle = torch.tensor(math.pi / 4)
R = torch.tensor([[torch.cos(angle), -torch.sin(angle)],
                   [torch.sin(angle),  torch.cos(angle)]])
print(f"Rotation:  R @ x = {R @ x}  (rotated 45 degrees)")

# Scaling
S = torch.tensor([[2.0, 0.0],
                   [0.0, 0.5]])
print(f"Scaling:   S @ x = {S @ x}  (stretched along x, shrunk along y)")

The key idea: **multiplying by a matrix mixes the coordinates together** according to specific rules. The question is: what kind of mixing is useful for quantization?

---
## 2. The Hadamard Matrix

The Hadamard matrix is a special matrix where **every entry is +1 or -1**. It is defined recursively:

$$H_1 = \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$$

$$H_n = \begin{bmatrix} H_{n-1} & H_{n-1} \\ H_{n-1} & -H_{n-1} \end{bmatrix}$$

Let's build them up.

In [ ]:
section_header("Building Hadamard Matrices", "Recursive construction")

# H_1: the simplest Hadamard matrix (2x2)
H1 = torch.tensor([[1.0, 1.0],
                    [1.0, -1.0]])
print("H_1 (2x2):")
print(H1)
print()

# H_2: built from H_1 using the recursive rule (4x4)
H2 = torch.cat([
    torch.cat([H1, H1], dim=1),
    torch.cat([H1, -H1], dim=1),
], dim=0)
print("H_2 (4x4):")
print(H2)
print()

# H_3: built from H_2 (8x8)
H3 = torch.cat([
    torch.cat([H2, H2], dim=1),
    torch.cat([H2, -H2], dim=1),
], dim=0)
print("H_3 (8x8):")
print(H3.int())

Notice the pattern: every entry is +1 or -1. No zeros, no fractions. Just additions and subtractions.

In [ ]:
section_header("Key Property: Orthogonality", "H @ H^T = d * I")

# For H_1 (d=2)
d1 = 2
print(f"H_1 @ H_1^T = ")
print(H1 @ H1.T)
print(f"Expected: {d1} * I = ")
print(d1 * torch.eye(d1))
print()

# For H_2 (d=4)
d2 = 4
product = H2 @ H2.T
print(f"H_2 @ H_2^T = {d2} * I? {torch.allclose(product, d2 * torch.eye(d2))}")

# For H_3 (d=8)
d3 = 8
product = H3 @ H3.T
print(f"H_3 @ H_3^T = {d3} * I? {torch.allclose(product, d3 * torch.eye(d3))}")

print("\nThis means H / sqrt(d) is an orthogonal matrix (preserves norms).")

**Why does orthogonality matter?** An orthogonal transform preserves distances and norms. If we transform a vector, quantize it, then inverse-transform, the quantization error in the original space equals the error in the transformed space. We don't lose anything by working in the transformed domain.

---
## 3. Why Hadamard? It Spreads Information

Imagine a vector where all the information is concentrated in one coordinate -- like `[1, 0, 0, 0, 0, 0, 0, 0]`. A scalar quantizer would waste 7 out of 8 quantization levels on zeros.

The Hadamard transform **spreads** that concentrated information evenly across ALL coordinates.

In [ ]:
section_header("Information Spreading", "One spike becomes uniform magnitude")

# A sparse vector: all info in one coordinate
sparse = torch.zeros(8)
sparse[0] = 1.0
print(f"Input (sparse):  {sparse.tolist()}")
print(f"  Non-zeros: {(sparse != 0).sum().item()} / 8")
print(f"  Magnitudes: {sparse.abs().tolist()}")
print()

# Multiply by normalized Hadamard
H3_normalized = H3 / math.sqrt(8)
spread = H3_normalized @ sparse
print(f"After H @ x:    {spread.tolist()}")
print(f"  Non-zeros: {(spread.abs() > 1e-6).sum().item()} / 8")
print(f"  Magnitudes: {spread.abs().tolist()}")
print(f"  All equal magnitude? {torch.allclose(spread.abs(), spread.abs()[0] * torch.ones(8))}")
print()

# Norm is preserved
print(f"  ||x||   = {sparse.norm():.4f}")
print(f"  ||H@x|| = {spread.norm():.4f}")
print("  Norms are the same -- no information lost, just redistributed.")

In [ ]:
# Visualize: before and after spreading
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Before: sparse vector", "After: H @ x (spread)"])

fig.add_trace(
    go.Bar(x=list(range(8)), y=sparse.tolist(), marker_color="steelblue",
           name="Input"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=list(range(8)), y=spread.tolist(), marker_color="coral",
           name="Transformed"),
    row=1, col=2
)

fig.update_layout(title="Hadamard Transform Spreads Information",
                  template="plotly_white", height=350, showlegend=False)
fig.update_yaxes(range=[-0.5, 1.1], row=1, col=1)
fig.update_yaxes(range=[-0.5, 1.1], row=1, col=2)
fig.show()

**This is exactly what we want for quantization.** If coordinates have similar magnitudes, a scalar quantizer can use its levels efficiently -- no wasted resolution on near-zero values.

---
## 4. The Problem: Matrix Multiply is O(d^2)

Computing `H @ x` with the full matrix requires `d * d` multiplications and additions.

For d = 1024, that's about 1 million operations per vector. For d = 4096 (common in LLMs), it's about 16 million. Can we do better?

In [ ]:
section_header("Cost of Naive Matrix Multiply")

for d in [8, 64, 256, 1024, 4096]:
    naive_ops = d * d
    fast_ops = d * int(math.log2(d))
    speedup = naive_ops / fast_ops
    print(f"d = {d:>5}: naive = {naive_ops:>12,} ops,  "
          f"butterfly = {fast_ops:>8,} ops,  "
          f"speedup = {speedup:.0f}x")

Yes! The **recursive structure** of H means we can compute H @ x in O(d log d) using a technique called the **butterfly**.

---
## 5. The Butterfly Trick

The key insight: because `H_n` is built from two copies of `H_{n-1}`, computing `H_n @ x` breaks into:
1. Split `x` into a left half and a right half
2. Compute: `new_left = old_left + old_right`, `new_right = old_left - old_right`
3. Recurse on each half

But instead of recursion, we can do it iteratively in `log2(d)` **stages**, working on pairs of increasing size.

For d = 8, there are 3 stages (since log2(8) = 3):
- **Stage 1** (h=1): pair up adjacent elements (distance 1)
- **Stage 2** (h=2): pair up elements at distance 2
- **Stage 3** (h=4): pair up elements at distance 4

In [ ]:
section_header("Butterfly Step by Step", "d=8, input = [1,0,0,0,0,0,0,0]")

x = torch.tensor([1.0, 0, 0, 0, 0, 0, 0, 0])
result = x.clone()

print(f"Input:   {result.tolist()}")
print()

# Stage 1: h=1, pairs of adjacent elements
h = 1
temp = result.view(-1, 2 * h)  # shape (4, 2)
left = temp[:, :h].clone()     # first element of each pair
right = temp[:, h:].clone()    # second element of each pair
temp = torch.cat([left + right, left - right], dim=1)
result = temp.view(-1)
print(f"Stage 1 (h=1): pair adjacent elements")
print(f"  {result.tolist()}")
print()

# Stage 2: h=2, pairs at distance 2
h = 2
temp = result.view(-1, 2 * h)  # shape (2, 4)
left = temp[:, :h].clone()     # first 2 of each group of 4
right = temp[:, h:].clone()    # last 2 of each group of 4
temp = torch.cat([left + right, left - right], dim=1)
result = temp.view(-1)
print(f"Stage 2 (h=2): pair elements at distance 2")
print(f"  {result.tolist()}")
print()

# Stage 3: h=4, pairs at distance 4
h = 4
temp = result.view(-1, 2 * h)  # shape (1, 8)
left = temp[:, :h].clone()     # first 4
right = temp[:, h:].clone()    # last 4
temp = torch.cat([left + right, left - right], dim=1)
result = temp.view(-1)
print(f"Stage 3 (h=4): pair elements at distance 4")
print(f"  {result.tolist()}")
print()

# Normalize
result_normalized = result / math.sqrt(8)
print(f"Normalized: {result_normalized.tolist()}")
print(f"Same as H3 @ x? {torch.allclose(result, H3 @ x)}")

In [ ]:
# Butterfly diagram visualization for d=8
# Shows the three stages of the FWHT butterfly network

d = 8
n_stages = int(math.log2(d))

fig = go.Figure()

# Layout: n_stages + 1 columns, d rows
col_spacing = 2.0
row_spacing = 1.0

# Draw nodes at each stage
for stage in range(n_stages + 1):
    cx = stage * col_spacing
    for row in range(d):
        cy = -row * row_spacing
        fig.add_trace(go.Scatter(
            x=[cx], y=[cy],
            mode="markers",
            marker=dict(size=12, color="steelblue"),
            showlegend=False,
            hoverinfo="skip",
        ))

# Label input and output
for row in range(d):
    cy = -row * row_spacing
    fig.add_annotation(x=-0.5, y=cy, text=f"x[{row}]",
                       showarrow=False, font=dict(size=11))
    fig.add_annotation(x=n_stages * col_spacing + 0.5, y=cy, text=f"y[{row}]",
                       showarrow=False, font=dict(size=11))

# Draw butterfly connections for each stage
colors = ["#e74c3c", "#27ae60", "#8e44ad"]
stage_labels = ["h=1", "h=2", "h=4"]

for stage in range(n_stages):
    h = 1 << stage  # 1, 2, 4
    x_from = stage * col_spacing
    x_to = (stage + 1) * col_spacing
    color = colors[stage % len(colors)]

    # Add stage label
    fig.add_annotation(
        x=(x_from + x_to) / 2, y=1.2,
        text=f"Stage {stage+1}<br>({stage_labels[stage]})",
        showarrow=False, font=dict(size=12, color=color)
    )

    # Process groups of 2*h
    for group_start in range(0, d, 2 * h):
        for i in range(h):
            top = group_start + i
            bot = group_start + i + h
            y_top = -top * row_spacing
            y_bot = -bot * row_spacing

            # Top output = top_in + bot_in  (solid line = addition)
            fig.add_trace(go.Scatter(
                x=[x_from, x_to], y=[y_top, y_top],
                mode="lines", line=dict(color=color, width=1.5),
                showlegend=False, hoverinfo="skip",
            ))
            fig.add_trace(go.Scatter(
                x=[x_from, x_to], y=[y_bot, y_top],
                mode="lines", line=dict(color=color, width=1.5),
                showlegend=False, hoverinfo="skip",
            ))

            # Bottom output = top_in - bot_in  (dashed line = subtraction)
            fig.add_trace(go.Scatter(
                x=[x_from, x_to], y=[y_bot, y_bot],
                mode="lines", line=dict(color=color, width=1.5, dash="dash"),
                showlegend=False, hoverinfo="skip",
            ))
            fig.add_trace(go.Scatter(
                x=[x_from, x_to], y=[y_top, y_bot],
                mode="lines", line=dict(color=color, width=1.5, dash="dash"),
                showlegend=False, hoverinfo="skip",
            ))

# Add legend annotations
fig.add_annotation(x=n_stages * col_spacing + 0.5, y=-d * row_spacing - 0.3,
                   text="Solid = add, Dashed = subtract",
                   showarrow=False, font=dict(size=10, color="gray"))

fig.update_layout(
    title="Butterfly Diagram: FWHT for d=8 (3 stages)",
    xaxis=dict(visible=False, range=[-1, n_stages * col_spacing + 1.5]),
    yaxis=dict(visible=False, scaleanchor="x"),
    template="plotly_white",
    height=500,
    width=700,
    margin=dict(l=60, r=60, t=60, b=40),
)
fig.show()

Each crossing in the diagram is one addition and one subtraction. There are `d/2` crossings per stage, and `log2(d)` stages, giving `d * log2(d) / 2` total operations. That is the butterfly.

---
## 6. Implement FWHT from Scratch

Now let's put it together. The loop walks through stages `h = 1, 2, 4, ..., d/2`, doing the same reshape-split-add/subtract pattern each time.

In [ ]:
def fwht(x):
    """Fast Walsh-Hadamard Transform.
    
    Operates on 1D vectors whose length is a power of 2.
    Returns the normalized transform (divided by sqrt(d)).
    """
    d = len(x)
    assert d > 0 and (d & (d - 1)) == 0, f"Length must be a power of 2, got {d}"
    
    result = x.clone().float()
    h = 1
    while h < d:
        # Reshape into groups of (2*h): each group has a left half and right half
        result = result.view(-1, 2 * h)
        left = result[:, :h].clone()
        right = result[:, h:].clone()
        # Butterfly: add and subtract
        result = torch.cat([left + right, left - right], dim=1)
        result = result.view(-1)  # flatten back
        h *= 2
    
    return result / math.sqrt(d)  # normalize for orthogonality


print("fwht defined. Let's test it!")

In [ ]:
section_header("Testing Our FWHT")

# Test 1: compare against matrix multiply
x = torch.tensor([1.0, 0, 0, 0, 0, 0, 0, 0])
our_result = fwht(x)
matrix_result = (H3 / math.sqrt(8)) @ x

check_close(our_result, matrix_result, name="FWHT vs matrix multiply")

In [ ]:
# Test 2: random vector
torch.manual_seed(42)
x_rand = torch.randn(8)
our_result = fwht(x_rand)
matrix_result = (H3 / math.sqrt(8)) @ x_rand

check_close(our_result, matrix_result, name="FWHT vs matrix (random input)")

---
## 7. Verify Properties

A correct FWHT should have three critical properties:

1. **Norm preservation**: `||fwht(x)|| == ||x||` (because it is an orthogonal transform)
2. **Self-inverse**: `fwht(fwht(x)) == x` (applying it twice gets you back)
3. **Deterministic**: same input always gives the same output

In [ ]:
section_header("Property 1: Norm Preservation", "||fwht(x)|| == ||x||")

torch.manual_seed(123)
for trial in range(5):
    x = torch.randn(16)
    y = fwht(x)
    check_close(y.norm(), x.norm(), name=f"Trial {trial+1}: norms")

In [ ]:
section_header("Property 2: Self-Inverse", "fwht(fwht(x)) == x")

torch.manual_seed(456)
for trial in range(5):
    x = torch.randn(16)
    y = fwht(x)
    x_reconstructed = fwht(y)
    check_close(x_reconstructed, x, name=f"Trial {trial+1}: roundtrip")

In [ ]:
section_header("Property 3: Deterministic", "Same input -> same output")

x = torch.tensor([3.0, 1, 4, 1, 5, 9, 2, 6])
y1 = fwht(x)
y2 = fwht(x)
y3 = fwht(x)

check_close(y1, y2, name="Run 1 vs Run 2")
check_close(y2, y3, name="Run 2 vs Run 3")

In [ ]:
section_header("Full Norm Preservation Test", "Using the helper")

# Use the course helper for a thorough test
torch.manual_seed(789)
test_vectors = torch.randn(100, 32)  # 100 vectors of dimension 32

# We need row-wise transforms, so wrap our 1D fwht for batched use
def fwht_batched(vecs):
    return torch.stack([fwht(v) for v in vecs])

norm_preservation_test(fwht_batched, fwht_batched, test_vectors, name="FWHT")

---
## 8. From Scratch to Production: The Reference Implementation

Our implementation above works on 1D vectors. But in practice, we need to transform **batches** of vectors -- for example, all the weight vectors in a layer at once.

Here is the actual `fwht` from the TurboQuant reference code (`reference-code/turboquant-torch/turboquant/hadamard.py`):

In [ ]:
def fwht_reference(x: torch.Tensor, normalize: bool = True) -> torch.Tensor:
    """Fast Walsh-Hadamard Transform (reference implementation).
    
    Operates on the LAST dimension. Handles arbitrary batch dimensions.
    This is the actual code from turboquant-torch.
    """
    d = x.shape[-1]
    if d <= 0 or (d & (d - 1)) != 0:
        raise ValueError(f"Last dim must be power of 2, got {d}")
    
    result = x.clone()
    h = 1
    while h < d:
        # Reshape to isolate pairs at distance h
        result = result.view(*result.shape[:-1], -1, 2 * h)
        left = result[..., :h]
        right = result[..., h:]
        result = torch.cat([left + right, left - right], dim=-1)
        result = result.view(*x.shape)
        h *= 2
    
    if normalize:
        result = result / math.sqrt(d)
    return result

### What changed?

The differences are small but important:

1. **`result.view(*result.shape[:-1], -1, 2 * h)`** -- instead of `.view(-1, 2*h)`, this preserves all batch dimensions. If the input is shape `(batch, seq_len, d)`, the reshape only touches the last dimension.

2. **`result[..., :h]` and `result[..., h:]`** -- the `...` (Ellipsis) means "all preceding dimensions, whatever they are." This makes the code work for any number of batch dimensions.

3. **`result.view(*x.shape)`** -- reshapes back to the original shape after each stage.

4. **`normalize` parameter** -- lets you skip normalization when you want the unnormalized Hadamard product (useful for combining with other transforms).

The butterfly logic is identical to what we wrote. The only additions handle generality.

In [ ]:
section_header("Verify: Our Implementation Matches Reference")

# 1D case
x1 = torch.randn(8)
check_close(fwht(x1), fwht_reference(x1), name="1D vector")

# Batch case: the reference handles batches natively
torch.manual_seed(42)
x_batch = torch.randn(5, 8)  # 5 vectors of dimension 8
ref_batch = fwht_reference(x_batch)
our_batch = torch.stack([fwht(row) for row in x_batch])
check_close(our_batch, ref_batch, name="Batched (5 vectors)")

# Multi-dimensional batch
x_3d = torch.randn(3, 4, 16)  # shape (3, 4, 16)
ref_3d = fwht_reference(x_3d)
check_shape(ref_3d, (3, 4, 16), name="3D batch output")
print("Reference handles arbitrary batch dims -- our loop-based version cannot.")

---
## Exercise

The WHT of `[1, 0, 0, 0, 0, 0, 0, 0]` (with normalization) should give a vector where **ALL** elements equal `1/sqrt(8)`. Compute it with your implementation and verify.

**Why?** The first row of the normalized Hadamard matrix is all `1/sqrt(d)`. So multiplying by the first standard basis vector picks out that row.

In [ ]:
section_header("Exercise: WHT of a Standard Basis Vector")

# Compute the FWHT of [1, 0, 0, 0, 0, 0, 0, 0]
e0 = torch.tensor([1.0, 0, 0, 0, 0, 0, 0, 0])
result = fwht(e0)

print(f"Input:    {e0.tolist()}")
print(f"FWHT(x):  {result.tolist()}")
print(f"Expected: {[1/math.sqrt(8)] * 8}")
print()

# Verify: all elements should equal 1/sqrt(8)
expected = torch.full((8,), 1.0 / math.sqrt(8))
check_close(result, expected, name="All elements = 1/sqrt(8)")

# Also verify the norm is preserved
check_close(result.norm(), e0.norm(), name="Norm preservation")

---
## Summary

| Concept | Key takeaway |
|---|---|
| Matrix multiply | Transforms vectors by mixing coordinates |
| Hadamard matrix | All +/-1 entries, orthogonal, recursive structure |
| Information spreading | Concentraded signals become uniform-magnitude -- ideal for scalar quantization |
| Butterfly trick | Exploits recursive structure for O(d log d) instead of O(d^2) |
| FWHT | Self-inverse, norm-preserving, deterministic |

**Next module**: We will combine the Hadamard transform with random sign flips to get the **Randomized Hadamard Transform (RHT)**, which makes any vector's coordinates look approximately Gaussian -- the perfect pre-processing step before scalar quantization.